# Evaluation — Italian WITS

Full evaluation with traditional metrics, abstraction metrics,
and LLM-as-Judge scoring.

**Memory strategy**: SigExt runs on CPU → unloaded → single LLM for both summary + judge.

In [1]:
!uv pip install -e ../..

Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Resolved 133 packages in 1.08s                                       
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip      
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
      Built sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
Prepared 1 package in 264ms                                              
Uninstalled 1 package in 0.99ms
Installed 1 package in 1ms file:///teamspace/studios/this_st
 ~ sm-sip==1.0.0 (from file:///teamspace/studios/this_studio/sm-sip)


In [2]:
from huggingface_hub import login
login()

## Configuration

In [3]:
from sm_sip.config import SigExtConfig, InferenceConfig, EvalConfig

sigext_config = SigExtConfig.from_preset("it", "10k-60t")
inference_config = InferenceConfig(lang="it", quantization="8bit", prompt_type="source_aware")
eval_config = EvalConfig(lang="it", judge_model_id="Qwen/Qwen2.5-14B-Instruct")

## Step 1: Load Data & Preprocess with SigExt (CPU)

In [ ]:
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, preprocess_dataset

test_data = get_test_data(lang="it", num_samples=100, skip_samples=sigext_config.skip_samples)

# SigExt runs on CPU to save GPU VRAM for the LLM
sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id, device="cpu")
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="it")

# Free memory before loading LLM
unload_sigext_model(sigext_model, sigext_tokenizer)
print(f"Preprocessed {len(processed_data)} samples.")

## Step 2: Load Single LLM (Summary + Judge)

In [ ]:
from sm_sip.models import load_llm, create_summary_chain, create_judge_chain
from sm_sip.prompts import get_summary_prompt, get_judge_prompt

# Single LLM for both tasks
llm_model, llm_tokenizer, gen_pipe = load_llm(
    inference_config.llm_model_id,
    inference_config.quantization,
    seed=inference_config.seed,
)

# Two chains, same underlying model
summary_chain = create_summary_chain(gen_pipe, get_summary_prompt("it", "source_aware"))
judge_chain = create_judge_chain(gen_pipe, get_judge_prompt("unified"))

## Step 3: Run Enhanced Evaluation

In [ ]:
from sm_sip.pipelines import run_enhanced_evaluation

metrics, samples = run_enhanced_evaluation(
    processed_data,
    summary_chain,
    judge_chain=judge_chain,
    lang="it",
)

print("\n=== RESULTS ===")
for key, val in metrics.items():
    print(f"  {key}: {val['mean']:.4f} ± {val['std']:.4f}")

## Step 4: Save Results

In [ ]:
from sm_sip.utils.io import save_results
from datetime import datetime

save_results({
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": sigext_config.model_id,
        "llm_model": inference_config.llm_model_id,
        "quantization": inference_config.quantization,
        "prompt_type": inference_config.prompt_type,
        "seed": inference_config.seed,
        "num_samples": len(samples),
    },
    "metrics": metrics,
    "samples": samples,
}, "results/italian/eval_enhanced.json")

print("Results saved!")

## Cleanup

In [ ]:
from sm_sip.utils.gpu import clear_gpu_memory
del llm_model, llm_tokenizer, gen_pipe
clear_gpu_memory()